In [1]:
import torch
import einops

In [2]:
emission_probs = [
    [
        [0.5, 0.1, 0.4],
        [0.3, 0.5, 0.2],
    ],
    [
        [0.4, 0.1, 0.5],
        [0.2, 0.5, 0.3],
    ]
]

In [3]:
emission_probs = torch.tensor(emission_probs)

In [9]:
selection = torch.tensor([0, 0])

In [10]:
emission_probs[selection, :]

tensor([[[0.5000, 0.1000, 0.4000],
         [0.3000, 0.5000, 0.2000]],

        [[0.5000, 0.1000, 0.4000],
         [0.3000, 0.5000, 0.2000]]])

In [11]:
# https://gist.github.com/EricCousineau-TRI/cc2dc27c7413ea8e5b4fd9675050b1c0
def vector_gather(vectors, indices):
    """
    Gathers (batched) vectors according to indices.
    Arguments:
        vectors: Tensor[N, L, D]
        indices: Tensor[N, K] or Tensor[N]
    Returns:
        Tensor[N, K, D] or Tensor[N, D]
    """
    N, L, D = vectors.shape
    squeeze = False
    if indices.ndim == 1:
        squeeze = True
        indices = indices.unsqueeze(-1)
    N2, K = indices.shape
    assert N == N2
    indices = einops.repeat(indices, "N K -> N K D", D=D)
    out = torch.gather(vectors, dim=1, index=indices)
    if squeeze:
        out = out.squeeze(1)
    return out

In [12]:
selected = vector_gather(emission_probs, selection)

In [13]:
selected

tensor([[0.5000, 0.1000, 0.4000],
        [0.4000, 0.1000, 0.5000]])

In [30]:
emission_probs[:, 0, 0] # shape is (2,)

tensor([0.5000, 0.4000])

In [32]:
dp = torch.zeros(2, 2, 3)

In [33]:
# assign the emission_probs to the upper left corners of dp
# for example, if emission_probs[:, 0, 0] is [0.5, 0.4], then
# dp[0][0][0] = 0.5 and dp[1][0][0] = 0.4
dp[:, 0, 0] = emission_probs[:, 0, 0]

In [34]:
dp

tensor([[[0.5000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000]],

        [[0.4000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000]]])